# Morningstar UK Dividend Yield Focus: June 2026 replication
### Using publicly available data, with a naïve Merton distance-to-default proxy

## 1. Configuration and input verification

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np


In [2]:
# Run from the project root.
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"

RAW_DIR = DATA_DIR / "raw"
FROZEN_DIR = RAW_DIR / "reference_2026-05-29"

INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
MANUAL_DIR = DATA_DIR / "manual"
VALIDATION_DIR = DATA_DIR / "validation"

LSE_FILE = RAW_DIR / "lse_may_2026.xlsx"
INSTRUMENT_FILE = RAW_DIR / "Instrument list.xlsx"
ANSWER_KEY_FILE = RAW_DIR / "answer_key.xlsx"
PREVIOUS_CONSTITUENTS_FILE = RAW_DIR / "pre-June 2026 constituents.xlsx"

YF_RAW_FILE = FROZEN_DIR / "yf_raw_2025-05-01_2026-05-29.pkl"
YF_ALT_FILE = FROZEN_DIR / "yf_raw_alt_currency.pkl"
FX_FILE = FROZEN_DIR / "fx_close.pkl"
FUNDAMENTALS_FILE = FROZEN_DIR / "fundamentals_pool.csv"
MANIFEST_FILE = FROZEN_DIR / "manifest.json"

for folder in [INTERIM_DIR, PROCESSED_DIR, MANUAL_DIR, VALIDATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [3]:
INPUT_FILES = {
    "lse": LSE_FILE,
    "instruments": INSTRUMENT_FILE,
    "answer_key": ANSWER_KEY_FILE,
    "previous_constituents": PREVIOUS_CONSTITUENTS_FILE,
    "prices": YF_RAW_FILE,
    "alternate_prices": YF_ALT_FILE,
    "fx": FX_FILE,
    "fundamentals": FUNDAMENTALS_FILE,
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not MANIFEST_FILE.is_file():
    raise FileNotFoundError(
        f"Missing manifest. Run the one-time setup first: {MANIFEST_FILE}"
    )

manifest = json.loads(
    MANIFEST_FILE.read_text(encoding="utf-8")
)

if manifest.get("status") != "approved":
    raise ValueError("The manifest has not been approved.")

verification = []

for name, path in INPUT_FILES.items():
    entry = manifest["files"].get(name)

    if entry is None:
        raise ValueError(f"No manifest entry for: {name}")

    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}")

    actual = sha256_file(path)

    if actual != entry["sha256"]:
        raise ValueError(
            f"Input has changed since it was frozen: {name}\n"
            f"File: {path}\n"
            "Check the change before proceeding."
        )

    verification.append({
        "input": name,
        "path": str(path.relative_to(BASE_DIR)),
        "sha256": actual,
        "verified": True,
    })

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(verification).to_csv(
    VALIDATION_DIR / "verified_inputs.csv",
    index=False,
)

print(f"Verified {len(verification)} input files against the manifest.")

Verified 8 input files against the manifest.


## 2. Load and inspect source data	

In [4]:
# Load the LSE issuer list (hash-checked in cell 4)
df = pd.read_excel(LSE_FILE, header=6)

# Remove the empty first column
df = df.iloc[:, 1:].copy()

print(df.head())
print("Rows:", df.shape[0])

  Admission Date           Company Name            ICB Industry  \
0     18/07/1994           3I GROUP PLC              Financials   
1     13/03/2007  3I INFRASTRUCTURE PLC              Financials   
2     17/02/2021           4BASEBIO PLC            Health Care    
3     13/03/1953     4IMPRINT GROUP PLC  Consumer Discretionary   
4     02/12/2013            80 MILE PLC         Basic Materials   

      ICB Super-Sector Country of Incorporation World Region        Market  
0  Financial Services            United Kingdom       Europe  MAIN MARKET   
1  Financial Services                    Jersey       Europe  MAIN MARKET   
2        Health Care             United Kingdom       Europe          AIM   
3                Media           United Kingdom       Europe  MAIN MARKET   
4      Basic Resources           United Kingdom       Europe          AIM   
Rows: 1534


In [5]:
print("Total records:", len(df))
print("Missing company names:",
      df["Company Name"].isna().sum())
print("Duplicate company names:",
      df["Company Name"].duplicated().sum())


Total records: 1534
Missing company names: 0
Duplicate company names: 0


In [6]:
df[" Company Name"] = (
    df["Company Name"]
    .str.strip()
    .str.replace(r"\s", " ", regex=True)
)
print (df["Company Name"].head(10))

0                          3I GROUP PLC
1                 3I INFRASTRUCTURE PLC
2                          4BASEBIO PLC
3                    4IMPRINT GROUP PLC
4                           80 MILE PLC
5                     88 ENERGY LIMITED
6                       AB DYNAMICS PLC
7               ABERDEEN ASIA FOCUS PLC
8    ABERDEEN ASIAN INCOME FUND LIMITED
9      ABERDEEN EQUITY INCOME TRUST PLC
Name: Company Name, dtype: str


In [7]:
print(df.columns.tolist())


['Admission Date', 'Company Name', 'ICB Industry', 'ICB Super-Sector', 'Country of Incorporation', 'World Region', 'Market', ' Company Name']


In [8]:

same_names = df["Company Name"].equals(df[" Company Name"])

print("All company names match:", same_names)

if same_names:
    df = df.drop(columns=[" Company Name"])

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)


All company names match: True
Columns: ['Admission Date', 'Company Name', 'ICB Industry', 'ICB Super-Sector', 'Country of Incorporation', 'World Region', 'Market']
Shape: (1534, 7)


In [9]:
# Load the LSE instrument list (hash-checked in cell 4)
instrument_path = INSTRUMENT_FILE

excel_file = pd.ExcelFile(instrument_path)
print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['1.0 All Equity', '1.1 Shares', '1.2 Depository Receipts', '1.3 ETFs', '1.4 Certificates', '1.5 Others', '2.0 All Non-Equity', '2.1 Bonds', '2.2 ETCs', '2.3 ETNs', '2.4 Securitised Derivatives', '2.5 SFPs', '2.6 Emission Allowances', 'Disclaimer']


In [10]:

# Load the Shares worksheet
shares = pd.read_excel(
    instrument_path,
    sheet_name="1.1 Shares",
    header=None
)

# Inspect the first 10 rows
print(shares.head(10))

# Check dataset dimensions
print("Rows:", shares.shape[0])
print("Columns:", shares.shape[1])


                             0                   1                2   \
0                           NaN                 NaN              NaN   
1                           NaN                 NaN              NaN   
2   Equity Instruments - Shares                 NaN              NaN   
3         As at 31 October 2026                 NaN              NaN   
4                           NaN                 NaN              NaN   
5                           NaN                 NaN              NaN   
6  Number of Instruments: 1,536                 NaN              NaN   
7                           NaN                 NaN              NaN   
8                          TIDM         Issuer Name  Instrument Name   
9                          1947  1947 OIL & GAS PLC      ORD GBP0.01   

             3                      4             5                      6   \
0           NaN                    NaN           NaN                    NaN   
1           NaN                    NaN           

In [11]:

# Reload the Shares worksheet with correct headers
shares = pd.read_excel(
    instrument_path,
    sheet_name="1.1 Shares",
    header=8
)

# Inspect the dataset
print(shares.head())

# Check column names
print("\nColumns:")
print(shares.columns.tolist())

# Check dimensions
print("\nShape:", shares.shape)


   TIDM            Issuer Name Instrument Name          ISIN  \
0  1947     1947 OIL & GAS PLC     ORD GBP0.01  GB00BRQMRP25   
1   III           3I GROUP PLC   ORD 73 19/22P  GB00B1YW4409   
2   3IN  3I INFRASTRUCTURE PLC         ORD NPV  JE00BF5FX167   
3   4BB           4BASEBIO PLC     ORD EUR1.00  GB00BMCLYF79   
4  FOUR     4IMPRINT GROUP PLC    ORD 38 6/13P  GB0006640972   

  MiFIR Identifier Code            ICB Industry ICB Super-Sector Name  \
0                  SHRS                  Energy                Energy   
1                  SHRS              Financials   Financial Services    
2                  SHRS              Financials   Financial Services    
3                  SHRS            Health Care          Health Care     
4                  SHRS  Consumer Discretionary                 Media   

   Start Date Country of Incorporation Trading Currency   LSE Market  \
0  28/09/2026           United Kingdom              GBX          AIM   
1  16/07/2007           United K

In [12]:

# Convert Start Date to a pandas date
shares["Start Date"] = pd.to_datetime(
    shares["Start Date"],
    dayfirst=True,
    errors="coerce"
)

# Define our historical snapshot
cutoff = pd.Timestamp("2026-05-31")

# Count instruments by start date
print("Started on or before cutoff:",
      (shares["Start Date"] <= cutoff).sum())

print("Started after cutoff:",
      (shares["Start Date"] > cutoff).sum())

print("Missing or invalid dates:",
      shares["Start Date"].isna().sum())


# Keep records with start dates on or before May 31
shares_historical = shares[
    shares["Start Date"] <= cutoff
].copy()

# Reset the row index
shares_historical = shares_historical.reset_index(drop=True)

# Check the result
print("Historical candidate instruments:", len(shares_historical))
print(shares_historical[["TIDM", "Issuer Name", "ISIN"]].head())

# Verify
print(shares_historical["Start Date"].max())


Started on or before cutoff: 1498
Started after cutoff: 38
Missing or invalid dates: 0
Historical candidate instruments: 1498
   TIDM            Issuer Name          ISIN
0   III           3I GROUP PLC  GB00B1YW4409
1   3IN  3I INFRASTRUCTURE PLC  JE00BF5FX167
2   4BB           4BASEBIO PLC  GB00BMCLYF79
3  FOUR     4IMPRINT GROUP PLC  GB0006640972
4   80M            80 MILE PLC  GB00BFD3VF20
2026-05-27 00:00:00


## 3. Build the eligible universe	

In [13]:

# Create a matching column in the May issuer dataset
df["match_name"] = (
    df["Company Name"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# Create the equivalent column in the instrument dataset
shares_historical["match_name"] = (
    shares_historical["Issuer Name"]
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# Check the results
print(df[["Company Name", "match_name"]].head())

print(shares_historical[["Issuer Name", "match_name"]].head())


# Verify 

print("May missing match names:", df["match_name"].isna().sum())
print("Instrument missing match names:",
      shares_historical["match_name"].isna().sum())

            Company Name             match_name
0           3I GROUP PLC           3I GROUP PLC
1  3I INFRASTRUCTURE PLC  3I INFRASTRUCTURE PLC
2           4BASEBIO PLC           4BASEBIO PLC
3     4IMPRINT GROUP PLC     4IMPRINT GROUP PLC
4            80 MILE PLC            80 MILE PLC
             Issuer Name             match_name
0           3I GROUP PLC           3I GROUP PLC
1  3I INFRASTRUCTURE PLC  3I INFRASTRUCTURE PLC
2           4BASEBIO PLC           4BASEBIO PLC
3     4IMPRINT GROUP PLC     4IMPRINT GROUP PLC
4            80 MILE PLC            80 MILE PLC
May missing match names: 0
Instrument missing match names: 0


In [14]:
matched = df["match_name"].isin(shares_historical["match_name"])
print("May issuers matched to an instrument:", matched.sum())
print("May issuers with NO instrument match:", (~matched).sum())
print(df.loc[~matched, ["Company Name", "Market"]].head(20).to_string(index=False))

May issuers matched to an instrument: 1389
May issuers with NO instrument match: 145
                           Company Name                          Market
                     AB "IGNITIS GRUPE"                    MAIN MARKET 
ABRDN DIVERSIFIED INCOME AND GROWTH PLC                    MAIN MARKET 
                    AEP PLANTATIONS PLC                    MAIN MARKET 
         AIR ASTANA JOINT STOCK COMPANY                    MAIN MARKET 
             ALBA MINERAL RESOURCES PLC                            AIM 
                 ALTONA RARE EARTHS PLC                    MAIN MARKET 
               ALUMINIUM BAHRAIN B.S.C.                    MAIN MARKET 
           AMEDEO AIR FOUR PLUS LIMITED                    MAIN MARKET 
                      ANA HOLDINGS INC.      ADMISSION TO TRADING ONLY 
                   ANIMALCARE GROUP PLC                            AIM 
                      AXIS BANK LIMITED PROFESSIONAL SECURITIES MARKET 
           BALTIC CLASSIFIEDS GROUP PLC            

In [15]:
def make_match_name(s):
    return s.str.strip().str.upper().str.replace(r"\s+", " ", regex=True)

# All equity instruments (shares + depositary receipts + others), any start date
all_equity = pd.read_excel(instrument_path, sheet_name="1.0 All Equity", header=8)
all_equity["match_name"] = make_match_name(all_equity["Issuer Name"])

shares["match_name"] = make_match_name(shares["Issuer Name"])

unmatched = df[~matched].copy()

# Default label, then overwrite in order of priority
unmatched["bucket"] = "not found - check manually"
unmatched.loc[unmatched["match_name"].isin(all_equity["match_name"]),
              "bucket"] = "in another equity sheet"
unmatched.loc[unmatched["match_name"].isin(shares["match_name"]),
              "bucket"] = "in shares, but start date after cutoff"

print(unmatched["bucket"].value_counts())


bucket
not found - check manually                69
in another equity sheet                   59
in shares, but start date after cutoff    17
Name: count, dtype: int64


In [16]:

# Count duplicated issuer matching keys
duplicate_count = shares_historical["match_name"].duplicated().sum()

print("Duplicate issuer keys:", duplicate_count)

# Show securities belonging to repeated issuer names
duplicates = shares_historical[
    shares_historical["match_name"].duplicated(keep=False)
]

print(
    duplicates[["Issuer Name", "TIDM", "ISIN"]].head(10)
)


Duplicate issuer keys: 88
                                   Issuer Name  TIDM          ISIN
13   ABERFORTH GEARED VALUE & INCOME TRUST PLC  AGVI  GB00BPJMQ253
14   ABERFORTH GEARED VALUE & INCOME TRUST PLC  AGZI  GB00BPJMQ360
78                             ANTOFAGASTA PLC  70GD  GB0000455849
79                             ANTOFAGASTA PLC  ANTO  GB0000456144
86              AQUILA EUROPEAN RENEWABLES PLC  AERI  GB00BK6RLF66
87              AQUILA EUROPEAN RENEWABLES PLC  AERS  GB00BK6RLF66
164                           BH MACRO LIMITED  BHMG  GG00BQBFY362
165                           BH MACRO LIMITED  BHMU  GG00BQBFY479
170                       BIOPHARMA CREDIT PLC  BPCP  GB00BDGKMY29
171                       BIOPHARMA CREDIT PLC  BPCR  GB00BDGKMY29


In [17]:

# Count distinct ISINs for each issuer
isin_counts = (
    shares_historical
    .groupby("match_name")["ISIN"]
    .nunique()
)

# Find issuers with more than one ISIN
multiple_isins = isin_counts[isin_counts > 1]

print("Issuers with multiple ISINs:", len(multiple_isins))

print(multiple_isins.head(10))


Issuers with multiple ISINs: 48
match_name
ABERFORTH GEARED VALUE & INCOME TRUST PLC    2
ANTOFAGASTA PLC                              2
BH MACRO LIMITED                             2
BOOT (HENRY) PLC                             2
BP PLC                                       3
BRAIME GROUP PLC                             2
BRUNNER INVESTMENT TRUST PLC                 2
CAB PAYMENTS HOLDINGS PLC                    3
CAFFYNS PLC                                  3
CELTIC PLC                                   3
Name: ISIN, dtype: int64


In [18]:
# Define preference-share flags before the diagnostic cells use them.
# PFD and ZDP are intentionally checked separately in later cells.
preference_mask = shares_historical["Instrument Name"].str.contains(
    r"PREF|\bPRF\b", case=False, na=False, regex=True
)
pref_shares = shares_historical.loc[preference_mask].copy()
print("Initial preference instruments:", len(pref_shares))

Initial preference instruments: 51


In [19]:

# Companies with multiple ISINs
multi_isin_issuers = set(multiple_isins.index)

# Companies with potential preference shares
preference_issuers = set(pref_shares["match_name"])

# Companies appearing in both groups
overlap = multi_isin_issuers.intersection(preference_issuers)

print("Multiple-ISIN issuers:", len(multi_isin_issuers))
print("Preference issuers:", len(preference_issuers))
print("Issuers in both groups:", len(overlap))


Multiple-ISIN issuers: 48
Preference issuers: 36
Issuers in both groups: 27


In [20]:

# Find multiple-ISIN issuers not flagged for preference shares
other_multi_isin_issuers = multi_isin_issuers - preference_issuers

print("Issuers requiring further investigation:",
      len(other_multi_isin_issuers))

# Display their securities
other_securities = shares_historical[
    shares_historical["match_name"].isin(other_multi_isin_issuers)
]

print(
    other_securities[
        ["Issuer Name", "TIDM", "Instrument Name", "ISIN"]
    ].head(20).to_string(index=False)
)


Issuers requiring further investigation: 21
                              Issuer Name TIDM                          Instrument Name         ISIN
ABERFORTH GEARED VALUE & INCOME TRUST PLC AGVI                                   ORD 1P GB00BPJMQ253
ABERFORTH GEARED VALUE & INCOME TRUST PLC AGZI                                   ZDP 1P GB00BPJMQ360
                         BH MACRO LIMITED BHMG                            ORD NPV (GBP) GG00BQBFY362
                         BH MACRO LIMITED BHMU                            ORD NPV (USD) GG00BQBFY479
                         BRAIME GROUP PLC  BMT                         'A'NON.V ORD 25P GB0001185270
                         BRAIME GROUP PLC BMTO                                  ORD 25P GB0001185056
                CAB PAYMENTS HOLDINGS PLC CABA   ORD 0.03P (ASSD MARA BIDCO CASH OFFER) GB00BV894922
                CAB PAYMENTS HOLDINGS PLC CABB ORD 0.03P (ASSD SHARE ALTERNATIVE OFFER) GB00BV88ZY11
                CAB PAYMENTS HOLDINGS PLC CABP 

In [21]:

# Find zero-dividend preference instruments
zdp_mask = shares_historical["Instrument Name"].str.contains(
    r"\bZDP\b",
    case=False,
    na=False,
    regex=True
)

print("ZDP instruments:", zdp_mask.sum())

# Check which ZDP records our earlier filter missed
new_zdp = shares_historical[zdp_mask & ~preference_mask]

print("Additional ZDP instruments:", len(new_zdp))

print(
    new_zdp[
        ["Issuer Name", "TIDM", "Instrument Name"]
    ].to_string(index=False)
)


ZDP instruments: 5
Additional ZDP instruments: 5
                              Issuer Name TIDM           Instrument Name
ABERFORTH GEARED VALUE & INCOME TRUST PLC AGZI                    ZDP 1P
                      EJF INVESTMENTS LTD EJFZ             2029 ZDP 100P
        EPE SPECIAL OPPORTUNITIES LIMITED ESOZ               ZDP 1P (DI)
                      UIL FINANCE LIMITED UTLH     2026 ZDP SHS 10P (DI)
                      UIL FINANCE LIMITED UTLI 2028 ZDP SHS 3.8676P (DI)


In [22]:

# Combine the preference and ZDP rules
shares_historical["is_preference"] = preference_mask | zdp_mask

# Count flagged securities
print("Potential preference instruments:",
      shares_historical["is_preference"].sum())

# Count other instruments
print("Other instruments:",
      (~shares_historical["is_preference"]).sum())

# Check total
print("Total instruments:", len(shares_historical))


Potential preference instruments: 56
Other instruments: 1442
Total instruments: 1498


In [23]:

# Create a provisional subset without flagged preference shares
ordinary_candidates = shares_historical[
    ~shares_historical["is_preference"]
].copy()

# Count remaining repeated issuer keys
duplicate_count = ordinary_candidates["match_name"].duplicated().sum()

# Count issuers with more than one instrument record
multiple_instrument_issuers = (
    ordinary_candidates.groupby("match_name")
    .size()
)

multiple_instrument_issuers = multiple_instrument_issuers[
    multiple_instrument_issuers > 1
]

print("Candidate instrument records:", len(ordinary_candidates))
print("Additional records from repeated issuer keys:", duplicate_count)
print("Issuers with multiple candidate records:", len(multiple_instrument_issuers))


Candidate instrument records: 1442
Additional records from repeated issuer keys: 45
Issuers with multiple candidate records: 37


In [24]:

# Select issuers with multiple remaining instruments
multi_candidates = ordinary_candidates[
    ordinary_candidates["match_name"].isin(
        multiple_instrument_issuers.index
    )
]

# Display examples
print(
    multi_candidates[
        [
            "Issuer Name",
            "TIDM",
            "Instrument Name",
            "ISIN",
            "Trading Currency"
        ]
    ].head(20).to_string(index=False)
)


                          Issuer Name TIDM                          Instrument Name         ISIN Trading Currency
       AQUILA EUROPEAN RENEWABLES PLC AERI                              ORD EUR0.01 GB00BK6RLF66              EUR
       AQUILA EUROPEAN RENEWABLES PLC AERS                        ORD EUR0.01 (GBP) GB00BK6RLF66              GBX
                     BH MACRO LIMITED BHMG                            ORD NPV (GBP) GG00BQBFY362              GBX
                     BH MACRO LIMITED BHMU                            ORD NPV (USD) GG00BQBFY479              USD
                 BIOPHARMA CREDIT PLC BPCP                              ORD USD0.01 GB00BDGKMY29              GBX
                 BIOPHARMA CREDIT PLC BPCR                              ORD USD0.01 GB00BDGKMY29              USD
                     BRAIME GROUP PLC  BMT                         'A'NON.V ORD 25P GB0001185270              GBX
                     BRAIME GROUP PLC BMTO                                  ORD 25P GB00

In [25]:

# Identify instrument descriptions containing PFD
pfd_mask = shares_historical["Instrument Name"].str.contains(
    r"\bPFD\b",
    case=False,
    na=False,
    regex=True
)

# Find instruments not already flagged
additional_pfd = shares_historical[
    pfd_mask & ~shares_historical["is_preference"]
]

print("Additional PFD instruments:", len(additional_pfd))

print(
    additional_pfd[
        ["Issuer Name", "TIDM", "Instrument Name", "ISIN"]
    ].to_string(index=False)
)


Additional PFD instruments: 2
                        Issuer Name TIDM            Instrument Name         ISIN
                         CELTIC PLC CCPC           CNV PFD ORD 100P GB0030639925
CITY OF LONDON INVESTMENT TRUST PLC BA47 PFD ORD STK(14% NON-CUM)#1 GB0001990059


In [26]:

# Add PFD instruments to our existing preference flag
shares_historical["is_preference"] = (
    shares_historical["is_preference"] | pfd_mask
)

# Recreate the ordinary-share candidate subset
ordinary_candidates = shares_historical[
    ~shares_historical["is_preference"]
].copy()

# Check counts
print("Potential preference instruments:",
      shares_historical["is_preference"].sum())

print("Remaining candidate instruments:",
      len(ordinary_candidates))

print("Total instruments:", len(shares_historical))


Potential preference instruments: 58
Remaining candidate instruments: 1440
Total instruments: 1498


In [27]:

print("Original instrument records:", len(shares))

print("After start-date filter:", len(shares_historical))

print("Potential preference securities flagged:",
      shares_historical["is_preference"].sum())

print("Remaining candidates:", len(ordinary_candidates))


Original instrument records: 1536
After start-date filter: 1498
Potential preference securities flagged: 58
Remaining candidates: 1440


In [28]:

# Find instrument records sharing the same ISIN
repeated_isins = ordinary_candidates[
    ordinary_candidates["ISIN"].duplicated(keep=False)
].copy()

print("Records with repeated ISINs:", len(repeated_isins))

print(
    repeated_isins[
        ["Issuer Name", "TIDM", "ISIN", "Trading Currency"]
    ].sort_values("ISIN").head(20).to_string(index=False)
)


Records with repeated ISINs: 37
                                    Issuer Name TIDM         ISIN Trading Currency
              DFI RETAIL GROUP HOLDINGS LIMITED  DFI BMG2624N1535              USD
              DFI RETAIL GROUP HOLDINGS LIMITED DFIB BMG2624N1535              USD
              DFI RETAIL GROUP HOLDINGS LIMITED DFIJ BMG2624N1535              USD
                     HONG KONG LAND HOLDINGS LD HKLJ BMG4587L1090              USD
                     HONG KONG LAND HOLDINGS LD HKLD BMG4587L1090              USD
                     HONG KONG LAND HOLDINGS LD HKLB BMG4587L1090              USD
                   JARDINE MATHESON HOLDINGS LD  JAR BMG507361001              USD
                   JARDINE MATHESON HOLDINGS LD JARJ BMG507361001              USD
                   JARDINE MATHESON HOLDINGS LD JARB BMG507361001              USD
                  GABELLI MERCHANT PARTNERS PLC  GMP GB00BD8P0741              USD
                  GABELLI MERCHANT PARTNERS PLC GMPP GB

In [29]:

# Count instruments by trading currency
currency_counts = ordinary_candidates[
    "Trading Currency"
].value_counts(dropna=False)

print(currency_counts)


Trading Currency
GBX    1372
USD      46
EUR      16
JPY       4
GBP       2
Name: count, dtype: int64


In [30]:
# Find records with repeated ISINs
repeated_isins = ordinary_candidates[
    ordinary_candidates["ISIN"].duplicated(keep=False)
].copy()

# Count repeated-ISIN records
print("Records with repeated ISINs:", len(repeated_isins))

# Display the relevant trading lines
print(
    repeated_isins[
        ["Issuer Name", "TIDM", "ISIN", "Trading Currency"]
    ].sort_values("ISIN").to_string(index=False)
)


Records with repeated ISINs: 37
                                    Issuer Name TIDM         ISIN Trading Currency
              DFI RETAIL GROUP HOLDINGS LIMITED  DFI BMG2624N1535              USD
              DFI RETAIL GROUP HOLDINGS LIMITED DFIB BMG2624N1535              USD
              DFI RETAIL GROUP HOLDINGS LIMITED DFIJ BMG2624N1535              USD
                     HONG KONG LAND HOLDINGS LD HKLJ BMG4587L1090              USD
                     HONG KONG LAND HOLDINGS LD HKLD BMG4587L1090              USD
                     HONG KONG LAND HOLDINGS LD HKLB BMG4587L1090              USD
                   JARDINE MATHESON HOLDINGS LD  JAR BMG507361001              USD
                   JARDINE MATHESON HOLDINGS LD JARJ BMG507361001              USD
                   JARDINE MATHESON HOLDINGS LD JARB BMG507361001              USD
                  GABELLI MERCHANT PARTNERS PLC  GMP GB00BD8P0741              USD
                  GABELLI MERCHANT PARTNERS PLC GMPP GB

In [31]:

# Count distinct securities among repeated-ISIN records
unique_isins = repeated_isins["ISIN"].nunique()

# Count additional trading-line records
extra_lines = len(repeated_isins) - unique_isins

print("Repeated-ISIN records:", len(repeated_isins))
print("Distinct ISINs involved:", unique_isins)
print("Additional trading lines:", extra_lines)


Repeated-ISIN records: 37
Distinct ISINs involved: 17
Additional trading lines: 20


In [32]:
# Admission date (company level) vs start date (instrument level)
df["Admission Date"] = pd.to_datetime(df["Admission Date"], dayfirst=True, errors="coerce")

compare = shares_historical.merge(
    df[["match_name", "Admission Date"]],
    on="match_name", how="inner"
)

later = compare["Start Date"] > compare["Admission Date"]
print("Matched lines:", len(compare))
print("Start Date later than company admission:", later.sum())
print("Same day:", (compare["Start Date"] == compare["Admission Date"]).sum())
print("Start Date earlier than admission:", (compare["Start Date"] < compare["Admission Date"]).sum())
print("Missing admission dates:", compare["Admission Date"].isna().sum())

Matched lines: 1476
Start Date later than company admission: 810
Same day: 602
Start Date earlier than admission: 64
Missing admission dates: 0


In [33]:
earlier = compare[compare["Start Date"] < compare["Admission Date"]]
print(earlier[["Issuer Name", "TIDM", "Start Date", "Admission Date", "LSE Market"]]
      .sort_values("Admission Date", ascending=False)
      .head(15).to_string(index=False))

                                 Issuer Name TIDM Start Date Admission Date  LSE Market
                        MAINTEL HOLDINGS PLC  MAI 2005-12-21     2016-04-28         AIM
                             SOUTH32 LIMITED  S32 2015-05-18     2015-05-26 MAIN MARKET
                            NANOCO GROUP PLC NANO 2005-03-24     2015-05-01 MAIN MARKET
                        AUTOTRADER GROUP PLC AUTO 2015-03-19     2015-03-24 MAIN MARKET
                           DFS FURNITURE PLC  DFS 2015-03-06     2015-03-11 MAIN MARKET
                                EUROCELL PLC ECEL 2015-03-04     2015-03-09 MAIN MARKET
                       WIZZ AIR HOLDINGS PLC WIZZ 2015-02-25     2015-03-02 MAIN MARKET
                   TAVISTOCK INVESTMENTS PLC TAVI 2014-06-02     2015-02-13         AIM
PROSERVICE BUILDING SERVICES MARKETPLACE PLC  PRO 2015-02-04     2015-02-09         AIM
       MORTGAGE ADVICE BUREAU (HOLDINGS) PLC MAB1 2014-11-12     2014-11-14 MAIN MARKET
                                

In [34]:
# --- Group A: the 17 whose share line starts after the cutoff ---
late_names = unmatched.loc[
    unmatched["bucket"] == "in shares, but start date after cutoff", "match_name"
]
late_detail = shares[shares["match_name"].isin(late_names)]

print("Group A - issuers with new share lines after cutoff:", late_names.nunique())
print(late_detail[["Issuer Name", "TIDM", "Instrument Name", "ISIN",
                   "Start Date", "LSE Market"]]
      .sort_values(["Issuer Name", "Start Date"]).to_string(index=False))

# --- Group B: the 69 not found ---
not_found = unmatched[unmatched["bucket"] == "not found - check manually"].copy()
not_found["Market"] = not_found["Market"].str.strip()

print("\nGroup B - not found:", len(not_found))
print(not_found["Market"].value_counts())

main_not_found = not_found[not_found["Market"].str.contains("MAIN MARKET", na=False)]
print("\nMain Market names:", len(main_not_found))
print(main_not_found[["Company Name", "ICB Industry", "Country of Incorporation"]]
      .to_string(index=False))

Group A - issuers with new share lines after cutoff: 17
                         Issuer Name TIDM    Instrument Name         ISIN Start Date  LSE Market
                 AEP PLANTATIONS PLC  AEP           ORD 2.5P GB00BWPD4D87 2026-06-25 MAIN MARKET
        BALTIC CLASSIFIEDS GROUP PLC  BCG        ORD GBP0.01 GB00BN44P254 2026-09-02 MAIN MARKET
      BLACKROCK SMALLER CO TRUST PLC BRSC             ORD 5P GB00BWQGKH23 2026-07-01 MAIN MARKET
           CAPITAL GEARING TRUST PLC  CGT       ORD GBP0.025 GB00BT66KP92 2026-07-23 MAIN MARKET
       CHRYSALIS INVESTMENTS LIMITED CHRY            ORD NPV GG00C0GC7Q38 2026-09-30 MAIN MARKET
GCP ASSET BACKED INCOME FUND LIMITED GABI            ORD NPV JE00BVMWGC49 2026-08-03 MAIN MARKET
                         GUNSYND PLC  GUN          ORD 0.03P GB00BXP3H253 2026-10-29         AIM
                         HELICAL PLC HLCL          ORD 1.05P GB00BT5GNW84 2026-07-20 MAIN MARKET
                 JOHNSON MATTHEY PLC JMAT   ORD 147 143/159P GB00BTQHVY

In [35]:
Path("data/interim").mkdir(parents=True, exist_ok=True)
main_not_found[["Company Name", "ICB Industry", "Country of Incorporation"]] \
    .to_csv("data/interim/group_b_main_market.csv", index=False)

In [36]:
keywords = ["BANK OF IRELAND", "BEAZLEY", "DCC", "FERGUSON", "FLUTTER",
            "HEADLAM", "PERSONAL FINANCE", "JTC", "PAGEGROUP", "PINEWOOD",
            "SCHRODER", "SENIOR", "SMURFIT", "TREATT", "WEIR", "MICHAEL PAGE"]

for kw in keywords:
    hits = all_equity[all_equity["Issuer Name"].str.contains(kw, case=False, na=False, regex=False)]
    if len(hits) == 0:
        print(f"{kw}: NOT IN OCTOBER FILE")
    else:
        print(f"{kw}: found")
        print(hits[["Issuer Name", "TIDM", "Instrument Name", "Start date"]].to_string(index=False))
    print("-" * 60)

BANK OF IRELAND: NOT IN OCTOBER FILE
------------------------------------------------------------
BEAZLEY: NOT IN OCTOBER FILE
------------------------------------------------------------
DCC: found
   Issuer Name TIDM   Instrument Name Start date
DCC ENERGY PLC  DCC ORD EUR0.25 (CDI) 08/07/2005
------------------------------------------------------------
FERGUSON: NOT IN OCTOBER FILE
------------------------------------------------------------
FLUTTER: NOT IN OCTOBER FILE
------------------------------------------------------------
HEADLAM: NOT IN OCTOBER FILE
------------------------------------------------------------
PERSONAL FINANCE: NOT IN OCTOBER FILE
------------------------------------------------------------
JTC: NOT IN OCTOBER FILE
------------------------------------------------------------
PAGEGROUP: NOT IN OCTOBER FILE
------------------------------------------------------------
PINEWOOD: NOT IN OCTOBER FILE
------------------------------------------------------------
SCH

In [37]:
Path("data/manual").mkdir(parents=True, exist_ok=True)

delisted = pd.DataFrame([
    # company,                  tidm,   left_lse,     reason,                   uk_flag
    ("BEAZLEY PLC",             "BEZ",  "2026-10-02", "takeover (Zurich)",      "uk"),
    ("SCHRODERS PLC",           "SDR",  "2026-10-02", "takeover (Nuveen)",      "uk"),
    ("SENIOR PLC",              "SNR",  "2026-10",    "takeover (Tinicum/Blackstone)", "uk"),
    ("JTC PLC",                 "JTC",  "2026-09-01", "takeover (Permira)",     "uk"),
    ("TREATT PLC",              "TET",  "2026-07-02", "takeover (Dohler)",      "uk"),
    ("INTERNATIONAL PERSONAL FINANCE PLC", "IPF", "2026-08-05", "takeover (BasePoint)", "uk"),
    ("PINEWOOD TECHNOLOGIES GROUP PLC", "PINE", "2026-10", "takeover (Ridgeview)", "uk"),
    ("HEADLAM GROUP PLC",       "HEAD", "2026-10",    "administration",         "uk"),
    ("BANK OF IRELAND GROUP PLC", "BIRG", "2026-06-29", "LSE delisting, Dublin primary", "review"),
    ("SMURFIT WESTROCK PLC",    "SWR",  "2026-06-22", "LSE delisting, NYSE primary", "review"),
    ("FERGUSON ENTERPRISES INC.", "FERG", "2026-07-20", "LSE delisting, NYSE primary", "review"),
    ("FLUTTER ENTERTAINMENT PLC", "FLTR", "2026-08-03", "LSE delisting, NYSE primary", "review"),
], columns=["company", "tidm", "left_lse", "reason", "uk_flag"])

delisted.to_csv("data/manual/delisted_after_cutoff.csv", index=False)
print(delisted)

                               company  tidm    left_lse  \
0                          BEAZLEY PLC   BEZ  2026-10-02   
1                        SCHRODERS PLC   SDR  2026-10-02   
2                           SENIOR PLC   SNR     2026-10   
3                              JTC PLC   JTC  2026-09-01   
4                           TREATT PLC   TET  2026-07-02   
5   INTERNATIONAL PERSONAL FINANCE PLC   IPF  2026-08-05   
6      PINEWOOD TECHNOLOGIES GROUP PLC  PINE     2026-10   
7                    HEADLAM GROUP PLC  HEAD     2026-10   
8            BANK OF IRELAND GROUP PLC  BIRG  2026-06-29   
9                 SMURFIT WESTROCK PLC   SWR  2026-06-22   
10           FERGUSON ENTERPRISES INC.  FERG  2026-07-20   
11           FLUTTER ENTERTAINMENT PLC  FLTR  2026-08-03   

                           reason uk_flag  
0               takeover (Zurich)      uk  
1               takeover (Nuveen)      uk  
2   takeover (Tinicum/Blackstone)      uk  
3              takeover (Permira)      uk  

In [38]:
sterling_isins = set(
    ordinary_candidates.loc[
        ordinary_candidates["Trading Currency"]
        .astype("string")
        .str.strip()
        .str.upper()
        .isin(["GBP", "GBX"]),
        "ISIN"
    ].dropna()
)

In [39]:
# Find ISINs without any sterling trading line
non_sterling_isins = repeated_isins[
    ~repeated_isins["ISIN"].isin(sterling_isins)
]

# Display their instrument details
print(
    non_sterling_isins[
        [
            "Issuer Name",
            "TIDM",
            "Instrument Name",
            "ISIN",
            "Trading Currency"
        ]
    ].sort_values("ISIN").to_string(index=False)
)


                      Issuer Name TIDM              Instrument Name         ISIN Trading Currency
DFI RETAIL GROUP HOLDINGS LIMITED  DFI     ORD $0.05 5/9 (SING.REG) BMG2624N1535              USD
DFI RETAIL GROUP HOLDINGS LIMITED DFIB ORD $0.05 5/9 (BERMUDA REGD) BMG2624N1535              USD
DFI RETAIL GROUP HOLDINGS LIMITED DFIJ   ORD $0.05 5/9 (JERSEY REG) BMG2624N1535              USD
       HONG KONG LAND HOLDINGS LD HKLB     ORD US$0.10(BERMUDA REG) BMG4587L1090              USD
       HONG KONG LAND HOLDINGS LD HKLD        ORD US$0.10(SING.REG) BMG4587L1090              USD
       HONG KONG LAND HOLDINGS LD HKLJ      ORD US$0.10(JERSEY REG) BMG4587L1090              USD
     JARDINE MATHESON HOLDINGS LD  JAR   ORD US$0.25(SINGAPORE REG) BMG507361001              USD
     JARDINE MATHESON HOLDINGS LD JARB     ORD US$0.25(BERMUDA REG) BMG507361001              USD
     JARDINE MATHESON HOLDINGS LD JARJ      ORD US$0.25(JERSEY REG) BMG507361001              USD


In [40]:
# Checking if the three issuers above are in May 2026 dataset

non_sterling_names = non_sterling_isins["match_name"].unique()
may_matches = df[
    df["match_name"].isin(non_sterling_names)
]
print("Matching May issuer records:", len(may_matches))
print(
    may_matches[
        ["Company Name", "Country of Incorporation", "Market"]
    ].to_string(index=False)
)


Matching May issuer records: 3
                     Company Name Country of Incorporation       Market
DFI RETAIL GROUP HOLDINGS LIMITED                  Bermuda MAIN MARKET 
       HONG KONG LAND HOLDINGS LD                  Bermuda MAIN MARKET 
     JARDINE MATHESON HOLDINGS LD                  Bermuda MAIN MARKET 


In [41]:

# Count issuers by country of incorporation
country_counts = df["Country of Incorporation"].value_counts(
    dropna=False
)

print(country_counts.head(15))

print("\nTotal issuer records:", len(df))


Country of Incorporation
United Kingdom            1133
Guernsey                    80
Jersey                      52
Ireland                     32
British Virgin Islands      30
Australia                   22
Bermuda                     20
Canada                      18
United States               18
Isle of Man                 14
Cayman Islands              13
India                       11
China                        8
Cyprus                       6
Japan                        5
Name: count, dtype: int64

Total issuer records: 1534


In [42]:

# Count companies by LSE market
market_counts = df["Market"].str.strip().value_counts(
    dropna=False
)

print(market_counts)
print("\nTotal:", market_counts.sum())


Market
MAIN MARKET                                907
AIM                                        604
ADMISSION TO TRADING ONLY                   14
PROFESSIONAL SECURITIES MARKET               7
MAIN MARKET / AIM                            1
MAIN MARKET / ADMISSION TO TRADING ONLY      1
Name: count, dtype: int64

Total: 1534


In [43]:

# Find companies with multiple market classifications
mixed_markets = df[
    df["Market"].str.contains("/", na=False, regex=False)
]

print(
    mixed_markets[
        ["Company Name", "Country of Incorporation", "Market"]
    ].to_string(index=False)
)


                     Company Name Country of Incorporation                                   Market
EPE SPECIAL OPPORTUNITIES LIMITED                  Bermuda                       MAIN MARKET / AIM 
                     INVESTEC PLC           United Kingdom MAIN MARKET / ADMISSION TO TRADING ONLY 


In [44]:
TAX_HAVENS = ["Jersey", "Guernsey", "Isle of Man", "British Virgin Islands",
              "Bermuda", "Cayman Islands", "Gibraltar", "Luxembourg"]

# Main Market only (includes the two mixed labels, e.g. "MAIN MARKET / AIM")
main = df[df["Market"].str.contains("MAIN MARKET", na=False)].copy()
inc = main["Country of Incorporation"].str.strip()

main["country_rule"] = "other - check"
main.loc[inc.isin(TAX_HAVENS), "country_rule"] = "tax haven -> primary listing"
main.loc[inc == "United Kingdom", "country_rule"] = "UK"

print("Main Market issuers:", len(main))
print(main["country_rule"].value_counts())
print(main.loc[main["country_rule"] == "other - check", "Country of Incorporation"]
          .value_counts().head(15))

Main Market issuers: 909
country_rule
UK                              652
tax haven -> primary listing    149
other - check                   108
Name: count, dtype: int64
Country of Incorporation
Ireland          18
United States     9
China             8
Australia         7
South Africa      4
Spain             4
Canada            4
Egypt             4
South Korea       4
Kazakhstan        3
Israel            3
France            3
Nigeria           3
Cyprus            3
Japan             3
Name: count, dtype: int64


In [45]:
others = main[main["country_rule"] == "other - check"].copy()

# Does the company have an ordinary share line (sheet 1.1) before the cutoff?
others["has_share_line"] = others["match_name"].isin(shares_historical["match_name"])

print(others["has_share_line"].value_counts())
print(others.loc[others["has_share_line"],
                 ["Company Name", "Country of Incorporation", "ICB Industry"]]
      .sort_values("Country of Incorporation").to_string(index=False))

has_share_line
True     64
False    44
Name: count, dtype: int64
                                    Company Name Country of Incorporation           ICB Industry
                           PYX RESOURCES LIMITED                Australia        Basic Materials
                         RESOLUTE MINING LIMITED                Australia        Basic Materials
                   NEW FRONTIER MINERALS LIMITED                Australia        Basic Materials
                                 SOUTH32 LIMITED                Australia        Basic Materials
                           GREENX METALS LIMITED                Australia        Basic Materials
                               BHP GROUP LIMITED                Australia        Basic Materials
                        BERKELEY ENERGIA LIMITED                Australia        Basic Materials
                CANADIAN GENERAL INVESTMENTS, LD                   Canada             Financials
                              ZENITH ENERGY LTD.              

In [46]:
others.loc[others["has_share_line"],
           ["Company Name", "Country of Incorporation", "ICB Industry"]] \
      .sort_values("Country of Incorporation") \
      .to_csv("data/interim/other_country_check.csv", index=False)

In [47]:
fca = (shares_historical[["match_name", "FCA Listing Category"]]
       .drop_duplicates("match_name"))

others_fca = others[others["has_share_line"]].merge(fca, on="match_name", how="left")

print(others_fca["FCA Listing Category"].value_counts(dropna=False))
print(others_fca[["Company Name", "Country of Incorporation", "FCA Listing Category"]]
      .sort_values("FCA Listing Category").to_string(index=False))

FCA Listing Category
Equity shares (international commercial companies secondary listing)    38
Equity shares (commercial companies)                                    16
Equity shares (transition)                                               8
Non-equity shares and non-voting equity shares                           1
NaN                                                                      1
Name: count, dtype: int64
                                    Company Name Country of Incorporation                                                 FCA Listing Category
                                   C&C GROUP PLC                  Ireland                                 Equity shares (commercial companies)
                     IRISH CONTINENTAL GROUP PLC                  Ireland                                 Equity shares (commercial companies)
                           KENMARE RESOURCES PLC                  Ireland                                 Equity shares (commercial companies)
      

In [48]:
# Used LLM for this 

fca = shares_historical[["match_name", "FCA Listing Category"]].drop_duplicates("match_name")
main = main.merge(fca, on="match_name", how="left")

# Full London listing, but main listing is at home
dual_home_primary = ["KERRY GROUP PLC", "AIB GROUP PLC", "IRISH CONTINENTAL GROUP PLC",
                     "THE MAGNUM ICE CREAM COMPANY N.V.",
                     "INTERNATIONAL CONSOLIDATED AIRLINES GROUP S.A."]
haven_not_uk = ["JARDINE MATHESON HOLDINGS LD", "HONG KONG LAND HOLDINGS LD",
                "DFI RETAIL GROUP HOLDINGS LIMITED"]

inc = main["Country of Incorporation"].str.strip()
full_london = main["FCA Listing Category"].eq("Equity shares (commercial companies)")

main["is_uk"] = (
    (inc == "United Kingdom")
    | (inc.isin(TAX_HAVENS) & ~main["match_name"].isin(haven_not_uk))
    | (full_london & ~main["match_name"].isin(dual_home_primary))
    | (main["match_name"] == "DCC PLC")      # renamed to DCC Energy; London primary
)

print("UK companies:", main["is_uk"].sum())
print("Coca-Cola HBC UK:", main.loc[main["match_name"] == "COCA-COLA HBC AG", "is_uk"].tolist())
print("Kerry UK:", main.loc[main["match_name"] == "KERRY GROUP PLC", "is_uk"].tolist())

UK companies: 810
Coca-Cola HBC UK: [True]
Kerry UK: [False]


In [49]:
uk = main[main["is_uk"]].copy()
print("UK companies:", len(uk))

# May name -> October name (found in your keyword search earlier)
RENAMES = {"DCC PLC": "DCC ENERGY PLC",
           "WEIR GROUP PLC": "WEIR PLC",
           "PAGEGROUP PLC": "MICHAEL PAGE PLC"}

# Check 1: are the May names really in our UK list?
for may_name in RENAMES:
    print(f"{may_name!r} in UK list:", may_name in set(uk["match_name"]))

uk["oct_name"] = uk["match_name"].replace(RENAMES)

# Check 2: are the October names really on the Shares sheet?
oct_names = set(shares["Issuer Name"].str.strip().str.upper()
                .str.replace(r"\s+", " ", regex=True))
for oct_name in RENAMES.values():
    print(f"{oct_name!r} on Shares sheet:", oct_name in oct_names)

UK companies: 810
'DCC PLC' in UK list: True
'WEIR GROUP PLC' in UK list: True
'PAGEGROUP PLC' in UK list: True
'DCC ENERGY PLC' on Shares sheet: True
'WEIR PLC' on Shares sheet: True
'MICHAEL PAGE PLC' on Shares sheet: True


In [50]:
lines = shares.copy()
print("All lines on the October Shares sheet:", len(lines))

# Same cleaning rule as match_name, so names line up with uk["oct_name"]
lines["oct_name"] = (lines["Issuer Name"].str.strip().str.upper()
                     .str.replace(r"\s+", " ", regex=True))
lines["TIDM"] = lines["TIDM"].str.strip()

# Flag preference shares and ZDPs (rule D14)
lines["is_preference"] = lines["Instrument Name"].str.contains(
    r"PREF|\bPRF\b|\bPFD\b|\bZDP\b", case=False, na=False)
print("Preference/ZDP lines:", lines["is_preference"].sum())

# Look at the non-equity category before deciding anything
non_equity = lines["FCA Listing Category"].eq("Non-equity shares and non-voting equity shares")
print("Non-equity / non-voting lines:", non_equity.sum())
print(lines.loc[non_equity & ~lines["is_preference"],
                ["Issuer Name", "TIDM", "Instrument Name", "LSE Market"]]
      .to_string(index=False))

All lines on the October Shares sheet: 1536
Preference/ZDP lines: 59
Non-equity / non-voting lines: 39
                     Issuer Name TIDM                          Instrument Name  LSE Market
HANSA INVESTMENT COMPANY LIMITED HANA                  'A' NON VTG ORD 1P (DI) MAIN MARKET
     NATIONWIDE BUILDING SOCIETY  NBS CORE CAPITAL DEFERRED SHS (MIN 250 CCDS) MAIN MARKET
        YOUNG & CO'S BREWERY PLC YNGN                        NON VTG ORD 12.5P MAIN MARKET


In [51]:
before = len(lines)
lines = lines[~lines["is_preference"] & ~non_equity].copy()
print("Lines removed:", before - len(lines))
print("Lines left:", len(lines))

Lines removed: 62
Lines left: 1474


In [52]:
CUTOFF = pd.Timestamp("2026-05-29")

lines["reissued_after_cutoff"] = lines["Start Date"] > CUTOFF

# For each company: does it have at least one line that existed at the cutoff?
has_pre = lines.groupby("oct_name")["reissued_after_cutoff"].transform(lambda s: (~s).any())

print("Lines starting after cutoff:", lines["reissued_after_cutoff"].sum())

# Drop a post-cutoff line only if the same company also has an older line
drop_mask = lines["reissued_after_cutoff"] & has_pre
print("Post-cutoff lines dropped (company has an older line):", drop_mask.sum())
lines = lines[~drop_mask].copy()

# Companies kept only through a re-issued line - which of them are in our UK list?
only_new = set(lines.loc[lines["reissued_after_cutoff"], "oct_name"])
in_uk = sorted(only_new & set(uk["oct_name"]))
print("Companies with only re-issued lines:", len(only_new), "| in UK universe:", len(in_uk))
print(in_uk)

Lines starting after cutoff: 37
Post-cutoff lines dropped (company has an older line): 1
Companies with only re-issued lines: 34 | in UK universe: 12
['AEP PLANTATIONS PLC', 'BALTIC CLASSIFIEDS GROUP PLC', 'BLACKROCK SMALLER CO TRUST PLC', 'CAPITAL GEARING TRUST PLC', 'CHRYSALIS INVESTMENTS LIMITED', 'GCP ASSET BACKED INCOME FUND LIMITED', 'HELICAL PLC', 'JOHNSON MATTHEY PLC', 'PACIFIC ASSETS TRUST PLC', 'PHOENIX SPREE DEUTSCHLAND LIMITED', 'PRISTINE CAPITAL PLC', 'TAYLOR MARITIME LIMITED']


In [53]:
lines["is_sterling"] = lines["Trading Currency"].isin(["GBX", "GBP"])

print("Lines before:", len(lines), "| distinct ISINs:", lines["ISIN"].nunique())

lines = (lines.sort_values(["ISIN", "is_sterling", "TIDM"], ascending=[True, False, True])
              .drop_duplicates(subset="ISIN", keep="first"))

print("Lines after:", len(lines))
print("Duplicate ISINs left:", lines["ISIN"].duplicated().sum())

# Spot-check two known cases
print(lines.loc[lines["oct_name"].str.contains("JARDINE|TAYLOR MARITIME"),
                ["oct_name", "TIDM", "ISIN", "Trading Currency"]].to_string(index=False))

Lines before: 1473 | distinct ISINs: 1452
Lines after: 1452
Duplicate ISINs left: 0
                    oct_name TIDM         ISIN Trading Currency
JARDINE MATHESON HOLDINGS LD  JAR BMG507361001              USD
     TAYLOR MARITIME LIMITED TMIP GG00C0G4QJ50              GBX


In [54]:
cols = ["oct_name", "TIDM", "ISIN", "Instrument Name", "Trading Currency",
        "FCA Listing Category", "reissued_after_cutoff"]

universe = (uk.drop(columns=["FCA Listing Category"], errors="ignore")
              .merge(lines[cols], on="oct_name", how="left"))

print("Rows:", len(universe))
print("Companies:", universe["match_name"].nunique())

no_line = universe[universe["TIDM"].isna()]
print("Companies with no share line:", len(no_line))

isins_per_co = universe.groupby("match_name")["ISIN"].nunique()
print("Companies with 2+ ISINs:", (isins_per_co > 1).sum())
print(isins_per_co[isins_per_co > 1].index.tolist())

no_line[["match_name", "ICB Industry", "Country of Incorporation"]] \
    .to_csv("data/interim/uk_no_share_line.csv", index=False)

Rows: 828
Companies: 810
Companies with no share line: 43
Companies with 2+ ISINs: 13
['BH MACRO LIMITED', 'CAB PAYMENTS HOLDINGS PLC', 'CT GLOBAL MANAGED PORTFOLIO TRUST PLC', 'CT UK HIGH INCOME TRUST PLC', 'CVC INCOME & GROWTH LIMITED', 'FIDELITY JAPAN TRUST PLC', 'MARWYN VALUE INVESTORS LIMITED', 'MITON UK MICROCAP TRUST PLC', 'NB DISTRESSED DEBT INVESTMENT FUND LIMITED', 'OXFORD TECHNOLOGY 2 VENTURE CAPITAL TRUST PLC', 'PACIFIC ASSETS TRUST PLC', 'SENECA GROWTH CAPITAL VCT PLC', 'SERAPHIM SPACE INVESTMENT TRUST PLC']


In [55]:
delisted = pd.read_csv("data/manual/delisted_after_cutoff.csv")

tidm_map = dict(zip(delisted["company"], delisted["tidm"]))

gap = universe["TIDM"].isna()
universe.loc[gap, "TIDM"] = universe.loc[gap, "match_name"].map(tidm_map)
universe["delisted_after_cutoff"] = universe["match_name"].isin(delisted["company"])

print("Filled from manual file:", universe["delisted_after_cutoff"].sum())
print(universe.loc[universe["delisted_after_cutoff"], ["match_name", "TIDM"]].to_string(index=False))

still_missing = universe[universe["TIDM"].isna()]
print("\nStill no ticker:", len(still_missing))
print(still_missing[["match_name", "ICB Industry"]].to_string(index=False))

Filled from manual file: 8
                        match_name TIDM
                       BEAZLEY PLC  BEZ
                 HEADLAM GROUP PLC HEAD
INTERNATIONAL PERSONAL FINANCE PLC  IPF
                           JTC PLC  JTC
   PINEWOOD TECHNOLOGIES GROUP PLC PINE
                     SCHRODERS PLC  SDR
                        SENIOR PLC  SNR
                        TREATT PLC  TET

Still no ticker: 35
                                      match_name           ICB Industry
         ABRDN DIVERSIFIED INCOME AND GROWTH PLC             Financials
                          ALTONA RARE EARTHS PLC        Basic Materials
                    AMEDEO AIR FOUR PLUS LIMITED            Industrials
                 BLACKROCK THROGMORTON TRUST PLC                    NaN
             BLUEFIELD SOLAR INCOME FUND LIMITED             Financials
                               BRISTOL WATER PLC              Utilities
                                   CARILLION PLC            Industrials
     CQS NATURAL

In [56]:
print(still_missing["match_name"].tolist())

['ABRDN DIVERSIFIED INCOME AND GROWTH PLC', 'ALTONA RARE EARTHS PLC', 'AMEDEO AIR FOUR PLUS LIMITED', 'BLACKROCK THROGMORTON TRUST PLC', 'BLUEFIELD SOLAR INCOME FUND LIMITED', 'BRISTOL WATER PLC', 'CARILLION PLC', 'CQS NATURAL RESOURCES GROWTH AND INCOME PLC', 'CT UK CAPITAL AND INCOME INVESTMENT TRUST PLC', 'DEFI DEVELOPMENT CORPORATION UK PLC', 'DIVERSE INCOME TRUST (THE) PLC', 'DP AIRCRAFT I LIMITED', 'ECCLESIASTICAL INSURANCE OFFICE PLC', 'EUROPEAN OPPORTUNITIES TRUST PLC', 'FISHGUARD & ROSSLARE RAILWAYS & HARBOURS COMPANY', 'GRESHAM HOUSE RENEWABLE ENERGY VCT 1 PLC', 'GRESHAM HOUSE RENEWABLE ENERGY VCT 2 PLC', 'INTUITIVE INVESTMENTS GROUP PLC', 'INVESTEC INVESTMENT TRUST PLC', 'JPMORGAN GLOBAL CORE REAL ASSETS LIMITED', 'MIDDLEFIELD CANADIAN INCOME PCC', 'NATIONAL WESTMINSTER BANK PLC', 'NATIONWIDE BUILDING SOCIETY', 'NB PRIVATE EQUITY PARTNERS LIMITED', 'NORTHERN ELECTRIC PLC', 'PICTON PROPERTY INCOME LD', 'PRAETURA GROWTH VCT PLC', 'RED CAPITAL PLC', 'RUFFER INVESTMENT COMPANY L

In [57]:
print(shares.loc[shares["Issuer Name"].str.contains("RUFFER|MIDDLEFIELD", na=False),
                 ["Issuer Name", "TIDM", "Instrument Name"]].to_string(index=False))

                    Issuer Name TIDM                Instrument Name
MIDDLEFIELD CANADIAN INCOME PCC MCTA RED PART PREF SHS NPV A RIGHTS
MIDDLEFIELD CANADIAN INCOME PCC MCTB RED PART PREF SHS NPV B RIGHTS
  RUFFER INVESTMENT COMPANY LTD RICA         RED PTG PREF SHS 0.01P


In [58]:
part = shares["Instrument Name"].str.contains(r"PTG PREF|PART PREF|PARTICIPATING PREF",
                                              case=False, na=False)
print("Participating-preference lines on the whole sheet:", part.sum())
print(shares.loc[part, ["Issuer Name", "TIDM", "Instrument Name", "LSE Market"]]
      .to_string(index=False))

Participating-preference lines on the whole sheet: 3
                    Issuer Name TIDM                Instrument Name  LSE Market
MIDDLEFIELD CANADIAN INCOME PCC MCTA RED PART PREF SHS NPV A RIGHTS MAIN MARKET
MIDDLEFIELD CANADIAN INCOME PCC MCTB RED PART PREF SHS NPV B RIGHTS MAIN MARKET
  RUFFER INVESTMENT COMPANY LTD RICA         RED PTG PREF SHS 0.01P MAIN MARKET


In [59]:
universe = universe[universe["TIDM"].notna()].copy()
print("Rows after dropping companies with no line:", len(universe),
      "| companies:", universe["match_name"].nunique())

re_mask = universe["ICB Industry"].str.strip().eq("Real Estate")
print("\nReal Estate rows:", re_mask.sum())
print(universe.loc[re_mask, "match_name"].tolist())

Rows after dropping companies with no line: 793 | companies: 775

Real Estate rows: 51
['ABRDN EUROPEAN LOGISTICS INCOME PLC', 'AEW UK REIT PLC', 'ALINA HOLDINGS PLC', 'ALTERNATIVE INCOME REIT PLC', 'ASEANA PROPERTIES LIMITED', 'BIG YELLOW GROUP PLC', 'BOOT (HENRY) PLC', 'BRITISH LAND COMPANY PLC', 'CARDIFF PROPERTY PLC', 'CLS HOLDINGS PLC', 'CUSTODIAN PROPERTY INCOME REIT PLC', 'DAR GLOBAL PLC', 'DERWENT LONDON PLC', 'FOXTONS GROUP PLC', 'GRAINGER PLC', 'GREAT PORTLAND ESTATES PLC', 'GRIT REAL ESTATE INCOME GROUP LIMITED', 'HAMMERSON PLC', 'HARWORTH GROUP PLC', 'HELICAL PLC', 'HOME REIT PLC', 'INTERNATIONAL WORKPLACE GROUP PLC', 'LAND SECURITIES GROUP PLC', 'LONDONMETRIC PROPERTY PLC', 'LSL PROPERTY SERVICES PLC', 'MEARS GROUP PLC', 'MOH NIPPON PLC', 'MOUNTVIEW ESTATES PLC', 'NEWRIVER REIT PLC', 'PALACE CAPITAL PLC', 'PHOENIX SPREE DEUTSCHLAND LIMITED', 'PLAZA CENTERS N.V.', 'PRIMARY HEALTH PROPERTIES PLC', 'REGIONAL REIT LIMITED', 'RESIDENTIAL SECURE INCOME PLC', 'RIGHTMOVE PLC', 'SA

In [60]:
real_estate = universe[re_mask].copy()
real_estate.to_csv("data/interim/real_estate_excluded.csv", index=False)

universe = universe[~re_mask].copy()
print("After REIT screen:", len(universe), "rows |", universe["match_name"].nunique(), "companies")

says_reit = real_estate["match_name"].str.contains("REIT|REAL ESTATE INVESTMENT TRUST")
print("Removed names that say REIT:", says_reit.sum(), "of", len(real_estate))

After REIT screen: 742 rows | 724 companies
Removed names that say REIT: 11 of 51


In [61]:
# 1. Flag closed-end funds
universe["is_closed_end_fund"] = universe["FCA Listing Category"].str.contains("Closed-ended", na=False)

# 2. Yahoo tickers
universe["yahoo"] = (universe["TIDM"].str.rstrip(".")
                     .str.replace(".", "-", regex=False) + ".L")
print("Tickers containing a dot:")
print(universe.loc[universe["TIDM"].str.contains(".", regex=False),
                   ["match_name", "TIDM", "yahoo"]].to_string(index=False))

# 3. Save
universe.to_csv(INTERIM_DIR / "universe_uk_main.csv", index=False)
print("\nSaved:", len(universe), "rows |",
      "closed-end funds:", universe["is_closed_end_fund"].sum(), "|",
      "delisted after cutoff:", universe["delisted_after_cutoff"].sum())


Tickers containing a dot:
                     match_name TIDM  yahoo
                   AO WORLD PLC  AO.   AO.L
ASHTEAD TECHNOLOGY HOLDINGS PLC  AT.   AT.L
                      AVIVA PLC  AV.   AV.L
                BAE SYSTEMS PLC  BA.   BA.L
                         BP PLC  BP.   BP.L
                   BT GROUP PLC BT.A BT-A.L
          JD SPORTS FASHION PLC  JD.   JD.L
              NATIONAL GRID PLC  NG.   NG.L
              QINETIQ GROUP PLC  QQ.   QQ.L
            R.E.A. HOLDINGS PLC  RE.   RE.L
                         RM PLC  RM.   RM.L
       ROLLS-ROYCE HOLDINGS PLC  RR.   RR.L
             SMITH & NEPHEW PLC  SN.   SN.L
              TAYLOR WIMPEY PLC  TW.   TW.L
     UNITED UTILITIES GROUP PLC  UU.   UU.L
                         VP PLC  VP.   VP.L

Saved: 742 rows | closed-end funds: 256 | delisted after cutoff: 8


## 4. Frozen prices, dividends and liquidity	

In [62]:
universe_file = INTERIM_DIR / "universe_uk_main.csv"

if not universe_file.is_file():
    raise FileNotFoundError(
        "Run the universe-building cells through original cell 61 first."
    )

universe = pd.read_csv(universe_file)
tickers = universe["yahoo"].dropna().unique().tolist()
if not YF_RAW_FILE.is_file():
    raise FileNotFoundError(
        f"Missing frozen prices: {YF_RAW_FILE}"
    )

# Run the manifest verification cell before this cell.
raw = pd.read_pickle(YF_RAW_FILE)
print("Loaded frozen price data.")

if not isinstance(raw.columns, pd.MultiIndex):
    raise ValueError("Expected (ticker, field) price columns.")

missing = set(tickers) - set(raw.columns.get_level_values(0))
if missing:
    raise ValueError(f"Frozen prices lack tickers: {sorted(missing)}")

close = raw.xs("Close", axis=1, level=1).reindex(columns=tickers)
no_data = close.columns[close.isna().all()].tolist()

print(
    f"With data: {len(tickers) - len(no_data)} "
    f"| No data: {len(no_data)}"
)
print(sorted(no_data))

Loaded frozen price data.
With data: 704 | No data: 38
['AC8.L', 'AERS.L', 'ALTR.L', 'BEZ.L', 'BPCP.L', 'CABA.L', 'CABB.L', 'DNE.L', 'EVR.L', 'FHP.L', 'FJVA.L', 'FJVB.L', 'HEAD.L', 'HID.L', 'HWC.L', 'IPF.L', 'JTC.L', 'MAC3.L', 'MACA.L', 'MINA.L', 'MINB.L', 'MNTS.L', 'NBDD.L', 'NBDG.L', 'NBDX.L', 'PACA.L', 'PACB.L', 'PINE.L', 'RNEP.L', 'SDR.L', 'SHPP.L', 'SNR.L', 'TET.L', 'TFGS.L', 'TORG.L', 'USFP.L', 'WGP.L', 'WRN.L']


In [63]:
nd = universe[universe["yahoo"].isin(no_data)].copy()

# Does the same company have another line that DID download?
has_data = set(universe.loc[~universe["yahoo"].isin(no_data), "match_name"])
nd["other_line_has_data"] = nd["match_name"].isin(has_data)

print(nd[["yahoo", "match_name", "Instrument Name", "Trading Currency",
          "is_closed_end_fund", "delisted_after_cutoff", "other_line_has_data"]]
      .sort_values(["delisted_after_cutoff", "is_closed_end_fund", "match_name"])
      .to_string(index=False))

nd.to_csv("data/interim/no_yahoo_data.csv", index=False)

 yahoo                                      match_name                          Instrument Name Trading Currency  is_closed_end_fund  delisted_after_cutoff  other_line_has_data
 AC8.L                           ACCELER8 VENTURES PLC                              ORD GBP0.01              GBX               False                  False                False
ALTR.L                               ALT RESOURCES PLC                             ORD GBP0.002              GBX               False                  False                False
CABB.L                       CAB PAYMENTS HOLDINGS PLC ORD 0.03P (ASSD SHARE ALTERNATIVE OFFER)              GBX               False                  False                 True
CABA.L                       CAB PAYMENTS HOLDINGS PLC   ORD 0.03P (ASSD MARA BIDCO CASH OFFER)              GBX               False                  False                 True
TORG.L              CHENAVARI TORO INCOME FUND LIMITED                            ORD NPV (GBP)              GBX   

In [64]:
ALT = {"AERS.L": "AERI.L", "BPCP.L": "BPCR.L", "USFP.L": "USF.L", "RNEP.L": "RNEW.L",
       "TFGS.L": "TFG.L",  "SHPP.L": "SHIP.L", "TORG.L": "TORO.L", "MNTS.L": "MNTN.L"}

alt_raw = pd.read_pickle(YF_ALT_FILE)
print("Loaded frozen alternate prices.")

for gbx, alt in ALT.items():
    h = alt_raw[alt].dropna(subset=["Close"])
    if h.empty:
        print(f"{gbx} -> {alt}: NO DATA")
        continue
    h.index = h.index.tz_localize(None)
    divs = h.loc[(h.index >= "2025-05-30") & (h.index <= "2026-05-29"), "Dividends"].sum()
    close = h.loc[h.index <= "2026-05-29", "Close"].iloc[-1]
    print(f"{gbx} -> {alt}: close={close:.4f}, 12m divs={divs:.4f}, yield={divs/close*100:.2f}%")

Loaded frozen alternate prices.
AERS.L -> AERI.L: close=0.1580, 12m divs=0.1650, yield=104.43%
BPCP.L -> BPCR.L: close=0.9580, 12m divs=0.0995, yield=10.38%
USFP.L -> USF.L: close=0.3670, 12m divs=0.0225, yield=6.13%
RNEP.L -> RNEW.L: close=0.1790, 12m divs=0.0711, yield=39.69%
TFGS.L -> TFG.L: close=12.6500, 12m divs=0.4600, yield=3.64%
SHPP.L -> SHIP.L: close=1.2250, 12m divs=0.0991, yield=8.09%
TORG.L -> TORO.L: close=0.5425, 12m divs=0.0634, yield=11.69%
MNTS.L -> MNTN.L: close=2.2500, 12m divs=0.0000, yield=0.00%


In [65]:
for t in ["AERI.L", "RNEW.L"]:
    h = alt_raw[t].dropna(subset=["Close"])
    h.index = h.index.tz_localize(None)
    print(t)
    print(h.loc[h["Dividends"] > 0, ["Close", "Dividends"]])

AERI.L
Price        Close  Dividends
Date                         
2025-05-22  0.6460   0.007974
2025-08-14  0.5250   0.008210
2025-11-13  0.3485   0.006837
2026-01-20  0.2700   0.094012
2026-03-23  0.2305   0.055944
RNEW.L
Price       Close  Dividends
Date                        
2026-05-13   0.17   0.071051


In [66]:
CUTOFF = pd.Timestamp("2026-05-29")
START = pd.Timestamp("2025-05-30")
SPECIAL = 0.10      # single payment > 10% of price = special / capital return

def get_hist(yahoo):
    """Price history for one line, using the other-currency ticker where needed."""
    src, tick = (alt_raw, ALT[yahoo]) if yahoo in ALT else (raw, yahoo)
    if tick not in src.columns.get_level_values(0):
        return None, tick
    h = src[tick].dropna(subset=["Close"])
    if h.empty:
        return None, tick
    if h.index.tz is not None:
        h.index = h.index.tz_localize(None)
    return h, tick

rows = []
for yahoo in universe["yahoo"].unique():
    h, tick = get_hist(yahoo)
    if h is None:
        rows.append({"yahoo": yahoo, "data_ticker": tick, "has_data": False})
        continue

    upto = h.loc[h.index <= CUTOFF]
    window = h.loc[(h.index >= START) & (h.index <= CUTOFF)]
    pays = window.loc[window["Dividends"] > 0, ["Close", "Dividends"]]
    is_special = (pays["Dividends"] / pays["Close"]) > SPECIAL

    close = upto["Close"].iloc[-1]
    div_all = pays["Dividends"].sum()
    div_regular = pays.loc[~is_special, "Dividends"].sum()

    rows.append({
        "yahoo": yahoo, "data_ticker": tick, "has_data": True,
        "price_date": upto.index.max().date(), "close": close,
        "n_payments": len(pays), "n_special": int(is_special.sum()),
        "div_12m_all": div_all, "div_12m_regular": div_regular,
        "yield_all": div_all / close * 100,
        "yield": div_regular / close * 100,
        "paid_dividend": div_regular > 0,
    })

yields = pd.DataFrame(rows)
yields.to_csv("data/interim/yields_ttm.csv", index=False)

In [67]:
check = ["RKT.L", "DNLM.L", "ADM.L", "TEP.L", "SGE.L", "PGOO.L"]

for t in check:
    h = raw[t].dropna(subset=["Close"])
    if h.index.tz is not None:
        h.index = h.index.tz_localize(None)
    pays = h.loc[h["Dividends"] > 0, ["Close", "Dividends"]].copy()
    pays["pct_of_price"] = (pays["Dividends"] / pays["Close"] * 100).round(2)
    print(t, "| close on 29 May:", round(h.loc[:"2026-05-29", "Close"].iloc[-1], 2))
    print(pays.to_string())
    print("-" * 50)

RKT.L | close on 29 May: 4591.0
Price         Close  Dividends  pct_of_price
Date                                        
2025-08-07  5693.75    87.9167          1.54
2026-02-02  6182.00   244.7917          3.96
2026-04-09  5180.00   127.8000          2.47
--------------------------------------------------
DNLM.L | close on 29 May: 780.5
Price        Close  Dividends  pct_of_price
Date                                       
2025-10-30  1124.0       28.0          2.49
2026-03-12   863.0       42.0          4.87
--------------------------------------------------
ADM.L | close on 29 May: 3296.0
Price        Close  Dividends  pct_of_price
Date                                       
2025-05-15  3266.0      121.0          3.70
2025-09-04  3444.0      115.0          3.34
2026-05-07  3178.0       90.0          2.83
--------------------------------------------------
TEP.L | close on 29 May: 1004.0
Price        Close  Dividends  pct_of_price
Date                                       
2025-07-24

In [68]:
fx = pd.read_pickle(FX_FILE)

fx.index = pd.to_datetime(fx.index, errors="raise")
if fx.index.tz is not None:
    fx.index = fx.index.tz_localize(None)

if fx.index.has_duplicates or not fx.index.is_monotonic_increasing:
    raise ValueError("Frozen FX dates must be unique and sorted.")

required_fx = {"GBPUSD=X", "EURUSD=X", "GEL=X"}
missing = required_fx - set(fx.columns)
if missing:
    raise ValueError(f"Missing frozen FX columns: {sorted(missing)}")

print("Loaded frozen FX.")

ADTV_START = pd.Timestamp("2026-03-01")

ALT_CCY = {"AERS.L": "EUR", "TORG.L": "EUR"}
line_ccy = dict(zip(universe["yahoo"], universe["Trading Currency"]))


def adtv_usd(yahoo):
    h, _ = get_hist(yahoo)
    if h is None:
        return None

    w = h.loc[
        (h.index >= ADTV_START) & (h.index <= CUTOFF),
        ["Close", "Volume"],
    ]
    if w.empty:
        return None

    ccy = (
        ALT_CCY.get(yahoo, "USD")
        if yahoo in ALT
        else line_ccy.get(yahoo)
    )

    rate = {
        "GBX": fx["GBPUSD=X"] / 100,
        "GBP": fx["GBPUSD=X"],
        "EUR": fx["EURUSD=X"],
    }.get(ccy)

    value = w["Close"] * w["Volume"]

    if ccy != "USD":
        if rate is None:
            return None

        aligned_rate = rate.dropna().reindex(w.index, method="ffill")
        if aligned_rate.isna().any():
            raise ValueError(f"Missing FX coverage for {yahoo}")

        value = value * aligned_rate

    return value.mean()


yields["adtv_usd"] = yields["yahoo"].apply(adtv_usd)
yields["adtv_pass"] = yields["adtv_usd"] >= 1_000_000

print("Lines with ADTV:", yields["adtv_usd"].notna().sum())
print("Pass $1m:", yields["adtv_pass"].sum())

added = ["NG.L", "III.L", "ICG.L", "BNZL.L", "QLT.L", "FOUR.L"]
print("\nJune additions:")
print(
    yields.loc[
        yields["yahoo"].isin(added),
        ["yahoo", "adtv_usd", "adtv_pass"],
    ].to_string(index=False)
)

yields.to_csv(INTERIM_DIR / "yields_ttm.csv", index=False)

Loaded frozen FX.
Lines with ADTV: 712
Pass $1m: 370

June additions:
 yahoo     adtv_usd  adtv_pass
 III.L 1.028802e+08       True
FOUR.L 3.923982e+06       True
BNZL.L 3.961618e+07       True
 ICG.L 3.653068e+07       True
  NG.L 2.524892e+08       True
 QLT.L 1.026898e+07       True


In [69]:
MAX_YIELD = 25.0   # yields above this are treated as data errors (D54)

yields["yield_error"] = yields["yield"] > MAX_YIELD
print("Lines flagged as yield errors:", yields["yield_error"].sum())
print(yields.loc[yields["yield_error"], ["yahoo", "yield"]]
      .sort_values("yield", ascending=False).to_string(index=False))

Lines flagged as yield errors: 7
 yahoo      yield
PGOO.L 604.395587
 BSV.L 536.912748
AAEV.L 533.023251
 TPV.L 465.116271
PEMB.L 380.434776
SVCT.L 340.909093
 NVT.L 263.157898


In [70]:
stage1 = universe.merge(yields, on="yahoo", how="left")

# Schroders: no Yahoo data after its October delisting.
# Documented exception: Morningstar's printed yield used (19-Jun-26 report).
sdr = stage1["yahoo"] == "SDR.L"
stage1.loc[sdr, "has_data"] = True
stage1.loc[sdr, "yield"] = 3.68
stage1.loc[sdr, "paid_dividend"] = True
stage1.loc[sdr, "yield_error"] = False

# Basic eligibility: paid a regular dividend and the yield isn't a data error
stage1["eligible"] = stage1["paid_dividend"].eq(True) & ~stage1["yield_error"].eq(True)

# One line per company: the most liquid share class (rulebook)
stage1 = (stage1.sort_values(["match_name", "adtv_usd"], ascending=[True, False])
                .drop_duplicates(subset="match_name", keep="first"))

print("Companies:", len(stage1))
print("With data:", stage1["has_data"].eq(True).sum())
print("Eligible (dividend paid, no yield error):", stage1["eligible"].sum())
print("Eligible AND ADTV >= $1m:", (stage1["eligible"] & stage1["adtv_pass"].eq(True)).sum())

Path("data/processed").mkdir(parents=True, exist_ok=True)
stage1.to_csv("data/processed/stage1_screened.csv", index=False)

Companies: 724
With data: 702
Eligible (dividend paid, no yield error): 452
Eligible AND ADTV >= $1m: 310


## 5. Baseline selection	

In [71]:
dividend_check = pd.DataFrame([
    ("SN.L",   29.04,  29.04,  "11.26p + 17.78p (sterling amounts of USD dividends)", "match"),
    ("ADM.L",  205.00, 205.00, "115.0p + 90.0p; incl. 29.1p + 17.2p special",           "match; specials included"),
    ("DNLM.L", 70.00,  70.00,  "28p + 17p + 25p special",                                "match; special included"),
    ("RKT.L",  460.51, 460.51, "special 235p x 25/24 = 244.79p (24-for-25 consolidation)", "match; special included"),
    ("TRIG.L", 7.55,   7.55,   "4 x 1.8875p",                                            "match (1 quarter verified)"),
    ("BATS.L", 241.44, 241.44, "3 x 60.06p + 1 x 61.26p",                                "match (60.06p unverified)"),
    ("CCH.L",  100.16, None,   "EUR 1.20 declared; FX rate differs from Morningstar",    "FX difference"),
], columns=["yahoo", "yahoo_div_12m", "rns_div_12m", "rns_detail", "result"])
Path("data/validation").mkdir(parents=True, exist_ok=True)
dividend_check.to_csv("data/validation/dividend_check.csv", index=False)
print(dividend_check.to_string(index=False))

 yahoo  yahoo_div_12m  rns_div_12m                                               rns_detail                     result
  SN.L          29.04        29.04      11.26p + 17.78p (sterling amounts of USD dividends)                      match
 ADM.L         205.00       205.00              115.0p + 90.0p; incl. 29.1p + 17.2p special   match; specials included
DNLM.L          70.00        70.00                                  28p + 17p + 25p special    match; special included
 RKT.L         460.51       460.51 special 235p x 25/24 = 244.79p (24-for-25 consolidation)    match; special included
TRIG.L           7.55         7.55                                              4 x 1.8875p match (1 quarter verified)
BATS.L         241.44       241.44                                  3 x 60.06p + 1 x 61.26p  match (60.06p unverified)
 CCH.L         100.16          NaN      EUR 1.20 declared; FX rate differs from Morningstar              FX difference


In [72]:
import pandas as pd


def select(pool, N=25):
    """Retain incumbents within the yield buffer, then fill remaining seats."""
    required = {"yahoo", "yield", "is_current"}
    missing = required - set(pool.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if not isinstance(N, int) or N <= 0:
        raise ValueError("N must be a positive integer")
    if pool[list(required)].isna().any().any():
        raise ValueError("Ticker, yield and incumbent flag cannot be missing")
    if pool["yahoo"].duplicated().any():
        raise ValueError("Duplicate tickers")
    if len(pool) < N:
        raise ValueError(f"Cannot select {N} stocks from {len(pool)} candidates")

    ranked = pool.sort_values(
        ["yield", "yahoo"], ascending=[False, True], kind="mergesort"
    ).reset_index(drop=True).copy()
    ranked["rank"] = ranked.index + 1

    incumbent = ranked["is_current"].eq(True)
    newcomer = ranked["is_current"].eq(False)
    within_buffer = ranked["rank"].le(int(1.33 * N))

    retained = ranked[incumbent & within_buffer].head(N)
    added = ranked[newcomer].head(N - len(retained))
    fallback = ranked[incumbent & ~within_buffer].head(
        N - len(retained) - len(added)
    )

    selected = pd.concat([
        retained.assign(status="retained"),
        added.assign(status="added"),
        fallback.assign(status="fallback_incumbent"),
    ]).sort_values("rank").reset_index(drop=True)

    assert len(selected) == N and selected["yahoo"].is_unique
    return selected, ranked


def _test_select():
    def fixture(incumbents, newcomers):
        tickers = (
            [f"I{i:03}.L" for i in range(incumbents)]
            + [f"N{i:03}.L" for i in range(newcomers)]
        )
        return pd.DataFrame({
            "yahoo": tickers,
            "yield": range(len(tickers), 0, -1),
            "is_current": [True] * incumbents + [False] * newcomers,
        })

    # Cap retained incumbents at 25.
    a, _ = select(fixture(35, 5))
    assert len(a) == 25 and a["yahoo"].is_unique
    assert a["status"].eq("retained").sum() == 25

    # Fill vacancies with newcomers.
    b, _ = select(fixture(6, 35))
    assert len(b) == 25 and b["status"].eq("added").sum() == 19

    # Small portfolio exercises fallback beyond its one-stock buffer.
    c, _ = select(fixture(2, 0), N=1)
    assert len(c) == 1 and c["status"].eq("retained").all()

    # Exactly 25 candidates, all newcomers.
    d, _ = select(fixture(0, 25))
    assert len(d) == 25 and d["status"].eq("added").all()

    # Reject undersized pools and duplicate tickers.
    duplicate = pd.concat(
        [fixture(0, 25), fixture(0, 1)], ignore_index=True
    )
    for invalid in [fixture(4, 5), duplicate]:
        try:
            select(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid pool was not rejected")

    print("Selection regression tests passed (6 scenarios).")


_test_select()

Selection regression tests passed (6 scenarios).


In [73]:
# Build the selection.
N = 25
prev = pd.read_excel(
    RAW_DIR / "pre-June 2026 constituents.xlsx", header=1
)
current = set(prev["ticker"].dropna())

s = stage1.copy()
s["is_current"] = s["yahoo"].isin(current)

selection_pool = s[
    s["eligible"].eq(True)
    & (s["is_current"] | s["adtv_pass"].eq(True))
].copy()

selected, ranked = select(selection_pool, N=N)

# Compare with the published constituents.
answer_key = pd.read_excel(ANSWER_KEY_FILE, header=1)
ms = set(answer_key["ticker"].dropna())
ours = set(selected["yahoo"].dropna())

print("Overlap:", len(ms & ours), "of", len(ms))
print("In both:", sorted(ms & ours))

rank_of = ranked.set_index("yahoo")["rank"].to_dict()
yield_of = s.set_index("yahoo")["yield"].to_dict()

print("\nMorningstar picked, we didn't (our rank / yield):")
for ticker in sorted(
    ms - ours,
    key=lambda t: (rank_of.get(t, float("inf")), t),
):
    rank = rank_of.get(ticker, "not ranked")
    dividend_yield = yield_of.get(ticker, float("nan"))
    print(
        f"  {ticker:8} rank {str(rank):>10}"
        f"   yield {dividend_yield:.2f}"
    )

print("\nWe picked, Morningstar didn't:")
extra = selected[~selected["yahoo"].isin(ms)]
columns = [
    "rank", "yahoo", "match_name", "yield",
    "is_closed_end_fund", "ICB Industry",
]
print(extra[[c for c in columns if c in extra.columns]].to_string(index=False))

Overlap: 4 of 25
In both: ['ASHM.L', 'DNLM.L', 'RKT.L', 'TRIG.L']

Morningstar picked, we didn't (our rank / yield):
  ADM.L    rank         51   yield 6.22
  BATS.L   rank         74   yield 5.26
  RAT.L    rank         81   yield 5.00
  FOUR.L   rank         83   yield 4.92
  EMG.L    rank         87   yield 4.62
  ICG.L    rank         89   yield 4.52
  SVT.L    rank         97   yield 4.24
  NG.L     rank        104   yield 4.06
  ULVR.L   rank        107   yield 4.02
  GRG.L    rank        108   yield 4.02
  SDR.L    rank        131   yield 3.68
  CRDA.L   rank        134   yield 3.65
  III.L    rank        148   yield 3.47
  IGG.L    rank        152   yield 3.42
  QLT.L    rank        161   yield 3.25
  BNZL.L   rank        167   yield 3.15
  HWDN.L   rank        183   yield 2.85
  SN.L     rank        198   yield 2.61
  CCEP.L   rank        200   yield 2.56
  SPX.L    rank        204   yield 2.44
  CCH.L    rank        211   yield 2.35

We picked, Morningstar didn't:
 rank  yaho

In [74]:
N = 25

# Pre-June constituents: only the ticker column is used as input.
prev = pd.read_excel(
    RAW_DIR / "pre-June 2026 constituents.xlsx", header=1
)
current = set(prev["ticker"].dropna())

s = stage1.copy()
s["is_current"] = s["yahoo"].isin(current)
print(
    "Pre-June constituents found in Stage 1:",
    s["is_current"].sum(), "of", len(current),
)

# Eligible incumbents, plus eligible newcomers passing ADTV.
selection_pool = s[
    s["eligible"].eq(True)
    & (s["is_current"] | s["adtv_pass"].eq(True))
].copy()

selected, ranked = select(selection_pool, N=N)

retained = selected[selected["status"].eq("retained")]
added = selected[selected["status"].eq("added")]
fallback_incumbents = selected[
    selected["status"].eq("fallback_incumbent")
]
dropped = s[
    s["is_current"] & ~s["yahoo"].isin(selected["yahoo"])
]

print("Ranked stocks:", len(ranked))
print(
    "Retained:", len(retained),
    "| Added:", len(added),
    "| Fallback:", len(fallback_incumbents),
    "| Total:", len(selected),
)
print("\nOur 25:")
print(
    selected[["rank", "yahoo", "match_name", "yield", "status"]]
    .to_string(index=False)
)
print("\nIncumbents we removed:")
print(
    dropped[["yahoo", "match_name", "yield"]]
    .to_string(index=False)
)

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
selected.to_csv(output_dir / "baseline_selection.csv", index=False)
ranked.to_csv(output_dir / "baseline_ranking.csv", index=False)

Pre-June constituents found in Stage 1: 25 of 25
Ranked stocks: 311
Retained: 5 | Added: 20 | Fallback: 0 | Total: 25

Our 25:
 rank  yahoo                                     match_name     yield   status
    1 NESF.L                  NEXTENERGY SOLAR FUND LIMITED 18.629834    added
    2  LRE.L                    LANCASHIRE HOLDINGS LIMITED 18.396526    added
    3  PAY.L                                   PAYPOINT PLC 16.497990    added
    4 SEIT.L               SDCL EFFICIENCY INCOME TRUST PLC 14.174109    added
    5  RCH.L                                      REACH PLC 13.542435    added
    6 FSFL.L                   FORESIGHT SOLAR FUND LIMITED 12.000000    added
    7 ENOG.L                                   ENERGEAN PLC 11.482582    added
    8  WPP.L                                        WPP PLC 11.324103    added
    9 BPCP.L                           BIOPHARMA CREDIT PLC 10.382046    added
   10  UKW.L                          GREENCOAT UK WIND PLC 10.367064    added
   1

In [75]:
fund_frozen = pd.read_csv(FUNDAMENTALS_FILE)

required = {
    "yahoo", "data_ticker", "fin_ccy", "px_ccy",
    "debt_period", "total_debt", "shares",
}

missing = required - set(fund_frozen.columns)
if missing:
    raise ValueError(f"Missing fundamentals columns: {sorted(missing)}")

if (
    fund_frozen["yahoo"].isna().any()
    or fund_frozen["yahoo"].duplicated().any()
):
    raise ValueError("Fundamentals need unique, non-missing tickers.")

fund_frozen["debt_period"] = pd.to_datetime(
    fund_frozen["debt_period"], errors="raise"
)

for column in ["total_debt", "shares"]:
    fund_frozen[column] = pd.to_numeric(
        fund_frozen[column], errors="raise"
    )

print("Loaded frozen fundamentals:", len(fund_frozen), "rows")

test = ["TEP.L", "RS1.L", "ITRK.L", "ICG.L", "FOUR.L"]
print(
    fund_frozen.loc[fund_frozen["yahoo"].isin(test)]
    .to_string(index=False)
)

Loaded frozen fundamentals: 311 rows
 yahoo data_ticker fin_ccy px_ccy debt_period   total_debt      shares
FOUR.L      FOUR.L     USD    GBp  2025-12-31    3400000.0  28045027.0
 ICG.L       ICG.L     GBP    GBp  2025-03-31 6106000000.0 289848928.0
ITRK.L      ITRK.L     GBP    GBp  2025-12-31 1648200000.0 153496203.0
 RS1.L       RS1.L     GBP    GBp  2025-03-31  511900000.0 468332550.0
 TEP.L       TEP.L     GBP    GBp  2025-03-31  194885000.0  79835390.0


6. Default-risk sensitivity analysis	

In [76]:
LAST_PERIOD = pd.Timestamp("2026-02-28")

pool = s[
    s["eligible"]
    & (s["is_current"] | s["adtv_pass"].eq(True))
].copy()

missing = set(pool["yahoo"]) - set(fund_frozen["yahoo"])
if missing:
    raise ValueError(
        f"Frozen fundamentals lack pool rows: {sorted(missing)}"
    )

fund = fund_frozen.loc[
    fund_frozen["yahoo"].isin(pool["yahoo"])
].copy()

expected_ticker = fund["yahoo"].map(lambda t: ALT.get(t, t))
if not fund["data_ticker"].eq(expected_ticker).all():
    raise ValueError("Fundamentals and price ticker mappings differ.")

if fund["debt_period"].gt(LAST_PERIOD).any():
    raise ValueError("Fundamentals contain disallowed debt periods.")

print("Pool size:", len(pool))
print("With total debt:", fund["total_debt"].notna().sum())
print("With shares:", fund["shares"].notna().sum())
print(
    "Accounts currencies:",
    fund["fin_ccy"].value_counts(dropna=False).to_dict(),
)

Pool size: 311
With total debt: 278
With shares: 311
Accounts currencies: {'GBP': 246, 'USD': 46, 'EUR': 9, nan: 8, 'GEL': 2}


In [77]:
# fx was loaded from the frozen FX file in cell 69.
def frozen_rate(column):
    window = fx.loc[
        (fx.index >= pd.Timestamp("2026-05-15"))
        & (fx.index <= CUTOFF),
        column,
    ].dropna()

    if window.empty:
        raise ValueError(f"No frozen {column} rate near cutoff.")

    value = float(window.iloc[-1])
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"Invalid frozen rate: {column}")

    return value


gbpusd = frozen_rate("GBPUSD=X")
eurusd = frozen_rate("EURUSD=X")

gel_window = fx.loc[
    (fx.index >= pd.Timestamp("2026-05-15"))
    & (fx.index <= CUTOFF),
    "GEL=X",
].dropna()

if gel_window.empty:
    gel_per_usd = np.nan
    print("GEL rate missing: GEL conversions and DD will be unavailable.")
else:
    gel_per_usd = frozen_rate("GEL=X")

TO_GBP = {
    "GBP": 1.0,
    "GBp": 0.01,
    "GBX": 0.01,
    "USD": 1 / gbpusd,
    "EUR": eurusd / gbpusd,
    "GEL": 1 / (gel_per_usd * gbpusd),
}

print(
    f"GBPUSD {gbpusd:.4f} | EURUSD {eurusd:.4f} "
    f"| GEL per USD {gel_per_usd:.4f}"
)


def vol_and_return(yahoo):
    h, _ = get_hist(yahoo)
    if h is None:
        return pd.Series({"vol": np.nan, "ret_12m": np.nan})

    c = h.loc[
        (h.index >= START) & (h.index <= CUTOFF),
        "Close",
    ].dropna()

    if len(c) < 2:
        return pd.Series({"vol": np.nan, "ret_12m": np.nan})

    return pd.Series({
        "vol": c.pct_change(fill_method=None).std() * np.sqrt(252),
        "ret_12m": c.iloc[-1] / c.iloc[0] - 1,
    })


# Use only the fundamentals columns needed here.
fund_columns = [
    "yahoo", "data_ticker", "fin_ccy", "px_ccy",
    "debt_period", "total_debt", "shares",
]

d = pool.merge(
    fund[fund_columns].drop(columns="data_ticker"),
    on="yahoo",
    how="left",
    validate="many_to_one",
)

d[["vol", "ret_12m"]] = d["yahoo"].apply(vol_and_return)

d["E"] = d["close"] * d["shares"] * d["px_ccy"].map(TO_GBP)
d["F"] = d["total_debt"] * d["fin_ccy"].map(TO_GBP)

sD = 0.05 + 0.25 * d["vol"]
V = d["E"] + d["F"]
sV = d["E"] / V * d["vol"] + d["F"] / V * sD

with np.errstate(divide="ignore", invalid="ignore"):
    d["dd"] = (
        np.log(V / d["F"])
        + (d["ret_12m"] - 0.5 * sV**2)
    ) / sV

# Preserve the original model's zero-debt treatment.
d.loc[d["F"] == 0, "dd"] = np.inf

# Percentile within sector: 0 = healthiest, 1 = weakest.
d["dd_pct"] = d.groupby("ICB Industry")["dd"].rank(
    pct=True, ascending=False
)

print("DD calculated:", d["dd"].notna().sum(), "of", len(d))

print("\nNo DD (missing input):")
print(
    d.loc[
        d["dd"].isna(),
        [
            "yahoo", "match_name", "is_closed_end_fund",
            "total_debt", "fin_ccy",
        ],
    ].to_string(index=False)
)

print("\nTest set:")
print(
    d.loc[
        d["yahoo"].isin(["TEP.L", "RS1.L", "ITRK.L", "ICG.L", "FOUR.L"]),
        ["yahoo", "ICB Industry", "E", "F", "vol", "dd", "dd_pct"],
    ].to_string(index=False)
)

d.to_csv(INTERIM_DIR / "dd_proxy.csv", index=False)

GEL rate missing: GEL conversions and DD will be unavailable.
GBPUSD 1.3444 | EURUSD 1.1653 | GEL per USD nan
DD calculated: 270 of 311

No DD (missing input):
 yahoo                                          match_name  is_closed_end_fund   total_debt fin_ccy
 AIE.L            ASHOKA INDIA EQUITY INVESTMENT TRUST PLC                True          NaN     GBP
BTRW.L                                  BARRATT REDROW PLC               False          NaN     GBP
 BWY.L                                         BELLWAY PLC               False          NaN     GBP
BPCP.L                                BIOPHARMA CREDIT PLC                True          NaN     USD
BRFI.L            BLACKROCK FRONTIERS INVESTMENT TRUST PLC                True          NaN     USD
CLDN.L                           CALEDONIA INVESTMENTS PLC               False          NaN     GBP
 CGT.L                           CAPITAL GEARING TRUST PLC                True          NaN     GBP
CVCG.L                         CVC INCOM

In [78]:
n_missing = (d["dd"].notna() & d["dd_pct"].isna()).sum()
assert n_missing == 0, f"{n_missing} stocks have DD but no ICB Industry percentile"
print("DD percentile check passed")

DD percentile check passed


In [79]:
d["dd_pass"] = (
    d["dd"].isna()                                        # no DD: passes untested
    | (~d["is_current"] & (d["dd_pct"] <= 0.50))          # newcomers: top 50%
    | ( d["is_current"] & (d["dd_pct"] <= 0.60))          # incumbents: top 60%
)
print("Pass DD proxy:", d["dd_pass"].sum(), "of", len(d))
print("4imprint:", d.loc[d["yahoo"] == "FOUR.L", ["dd", "dd_pct", "dd_pass"]].to_string(index=False))

merton_sel, merton_rank = select(d[d["dd_pass"]])
ours_m = set(merton_sel["yahoo"])

print("\nOverlap with Morningstar:", len(ms & ours_m), "of 25  (baseline: 4)")
print("In both:", sorted(ms & ours_m))
print("Still extra:", sorted(ours_m - ms))

merton_sel.to_csv("data/processed/merton_dd_selection.csv", index=False)

Pass DD proxy: 176 of 311
4imprint:        dd  dd_pct  dd_pass
16.322375    0.05     True

Overlap with Morningstar: 5 of 25  (baseline: 4)
In both: ['ASHM.L', 'BATS.L', 'DNLM.L', 'RKT.L', 'TRIG.L']
Still extra: ['BIPS.L', 'BPCP.L', 'CVCG.L', 'FGEN.L', 'FSFL.L', 'GCP.L', 'GKP.L', 'HFEL.L', 'MONY.L', 'NCYF.L', 'NESF.L', 'ORIT.L', 'PEY.L', 'SBRE.L', 'SEIT.L', 'SEQI.L', 'SMIF.L', 'TBCG.L', 'TFIF.L', 'TW.L']


In [80]:
is_fund = d["is_closed_end_fund"].eq(True)

new_thr = np.where(is_fund, 0.30, 0.50)    # newcomers: no-moat 30%, otherwise 50%
cur_thr = np.where(is_fund, 0.36, 0.60)    # incumbents: no-moat 36%, otherwise 60%
thr = np.where(d["is_current"], cur_thr, new_thr)

d["dd_pass_v2"] = np.where(
    d["dd"].isna(),
    ~is_fund,                       # no DD: operating companies pass untested, funds fail
    d["dd_pct"] <= thr,
)
print("Pass v2:", d["dd_pass_v2"].sum(), "of", len(d))

v2_sel, v2_rank = select(d[d["dd_pass_v2"]])
ours_v2 = set(v2_sel["yahoo"])

print("\nOverlap with Morningstar:", len(ms & ours_v2), "of 25  (baseline 4, Merton 5)")
print("In both:", sorted(ms & ours_v2))
print("Still extra:", sorted(ours_v2 - ms))
print("Morningstar's, still missing:", sorted(ms - ours_v2))

v2_sel.to_csv("data/processed/merton_dd_moatrule_selection.csv", index=False)

Pass v2: 132 of 311

Overlap with Morningstar: 7 of 25  (baseline 4, Merton 5)
In both: ['ASHM.L', 'BATS.L', 'DNLM.L', 'FOUR.L', 'RAT.L', 'RKT.L', 'ULVR.L']
Still extra: ['BIPS.L', 'BP.L', 'BTRW.L', 'BYIT.L', 'CRE.L', 'GCP.L', 'GKP.L', 'HFD.L', 'IHP.L', 'ITV.L', 'MONY.L', 'NXT.L', 'PETS.L', 'PSN.L', 'SBRE.L', 'TBCG.L', 'TW.L', 'WIL.L']
Morningstar's, still missing: ['ADM.L', 'BNZL.L', 'CCEP.L', 'CCH.L', 'CRDA.L', 'EMG.L', 'GRG.L', 'HWDN.L', 'ICG.L', 'IGG.L', 'III.L', 'NG.L', 'QLT.L', 'SDR.L', 'SN.L', 'SPX.L', 'SVT.L', 'TRIG.L']
